# 集群通信介绍

本节回答本章的第一个问题：**集群训练为什么离不开集合通信？**

我们先看清单卡训练撞上的两堵墙，再看分布式训练引入了什么新问题，
最后落到四类常用并行方式与 MoE 的 All2All 通信。

## 本节学习大纲

- 为什么需要集群训练：性能与内存的双重瓶颈
- 集群训练与集合通信
- 集群训练有哪些常用并行方式（含「专家并行在 MoE 中怎么工作」的举例）
- 课堂小练
- 本节总结

## 1. 为什么需要集群训练：性能与内存的双重瓶颈

为什么不能只用一张卡训练大模型？答案在于**两个同时到来的瓶颈**。

### 1.1 瓶颈一：性能瓶颈——单卡训练周期不可接受

单卡算力已无法满足训练时效性需求。

> 根据业界论文理论推算，端到端训练 AI 大模型的理论时间满足：
>
> $$E_t \approx \frac{8 \cdot T \cdot P}{n \cdot X}$$
>
> 其中：
> - $E_t$ — 端到端训练时间（End-to-end time）
> - $T$ — 训练 token 总数
> - $P$ — 模型参数量
> - $n$ — 并行度（使用的卡数）
> - $X$ — 单卡算力
>
> 该式直观地说明：**训练时间与参数量、数据量成正比，与并行度、单卡算力成反比。** 当 $P$ 和 $T$
> 增长到万亿级规模时，仅靠提升单卡算力 $X$ 已经追不上需求，必须提升并行度 $n$。

以 GPT-4 量级模型为例（约 1.8 T 参数 + 13 T 数据）：

| 配置 | 训练周期 | 说明 |
| --- | --- | --- |
| 25000 卡 A100 集群 | **~90 天**（完成训练） | 业界顶级集群规模 |

**结论**：即便是 25000 卡的顶级集群，完成一次训练仍需约 90 天。若把卡数退化到单卡，
训练周期将变得完全不可接受。

### 1.2 瓶颈二：内存挑战——模型加载即为难题

单卡的内存容量，甚至装不下模型的参数本身。

| 项目 | 数值 |
| --- | --- |
| GPT-4 参数量 | 约 1.8 万亿（1.8 T） |
| 参数占用显存 | 约 **1.8 TB** |
| 参数全部加载所需卡数 | **~128 卡** |

**结论**：GPT-4 有 1.8 万亿参数规模，参数占用约 1.8 TB，仅模型权重本身就已无法放入单张卡，
需要约 128 张卡才能装下参数。

### 1.3 破局思路：分布式集群训练与并行计算

> 面对超大参数和超大数据量模型，单卡已触及物理极限。
>
> **使用分布式集群训练与并行计算技术，将数据与模型切分至多张计算卡，突破性能与内存瓶颈。**

但切分本身会带来新的代价：

> **不管切数据还是切模型，都需要大量的数据通信。**

这正是集合通信存在的意义——它是把多张卡「拧成一台逻辑机器」的粘合剂。

### 1.4 两个瓶颈的对照

| 维度 | 性能瓶颈 | 内存挑战 |
| --- | --- | --- |
| 问题本质 | 单卡算力不足以在合理时间内完成训练 | 单卡容量不足以装下模型参数 |
| 量化示例 | 25000 卡仍需 ~90 天 | 1.8T 参数 ≈ 1.8TB，约需 128 卡 |
| 解决方向 | 提高并行度 $n$（切数据 / 切模型） | 切分模型参数到多卡 |
| 引入的新问题 | — | **切分后需要大量数据通信** |

### 1.5 新问题：通信本身成了瓶颈

切分解决了「单卡算不动、装不下」，却把**通信**推到了台前。华为官方在 2026 年全联接大会上
给过一组数字：

> 在传统计算架构下，集群规模扩大时，资源利用率反而下降，**十万卡集群的实际模型算力利用率
> 仅约 20%**，大量算力处于**等待通信完成**的状态。

换句话说：**卡越多，通信越容易拖后腿**。买回来的算力有多少能真正用在计算上，很大程度取决于
通信做得好不好。这正是集合通信库（HCCL）存在的意义——它不是「锦上添花」，而是决定一个集群
能不能跑起来的关键一环。

## 2. 集群训练与集合通信

把数据或模型切分到多张 NPU 之后，各卡各算一部分，但训练过程需要它们的**结果保持一致**。

### 2.1 切分数据的情形（数据并行）

```
   Batch1      Batch2      Batch3
     │           │           │
   NPU1        NPU2        NPU3        ← 每张卡拿不同的数据子集
     │           │           │
  前向传播     前向传播     前向传播
     │           │           │
  Loss 计算    Loss 计算    Loss 计算
     │           │           │
  反向传播     反向传播     反向传播
     │           │           │
     └───────────┴───────────┘
                 │
        对梯度做 AllReduce        ← 关键步骤
                 │
     ┌───────────┴───────────┐
  更新权重     更新权重     更新权重
```

每张卡用不同的数据算出一个梯度，但模型的**权重只有一份逻辑副本**。如果各卡直接用自己算出的
梯度更新权重，三张卡的权重就会立刻发散，模型彻底失效。

因此必须在更新权重之前，让所有卡的梯度**达成一致**——这正是 **AllReduce** 的工作：
对多卡梯度做归约（通常求和后取平均），再把一致的结果广播回所有卡。

![图 1](./images/png/allreduce_gradient_sync.png)

*图 1 · AllReduce 逐元素求和的逻辑输入输出。箭头表示数据关系，具体交换步骤（分块、链路）由通信算法决定。*

> **算子规定「结果」，算法规定「过程」。** AllReduce 规定「归约结果由所有 Rank 获得」；
> 至于走哪条链路、如何分块、分几步交换，则由具体通信算法决定。这一点将在第 2 章展开。

### 2.2 一次 AllReduce 到底要搬多少数据

既然通信这么关键，它到底有多「重」？可以粗略算一笔账：

- **梯度有多大**：梯度覆盖模型的全部参数；参数量记作 P，用 fp32 存储时占 **4P 字节**；
- **AllReduce 要搬多少**：以最常用的 Ring 算法为例，**每张卡**要收发的数据量约为
  **2 × (N−1)/N × 梯度大小**（N 为卡数）；N 较大时，**接近 2 倍梯度大小**。

以 **7B 模型**为例：

| 项目 | 数值 |
| --- | --- |
| 参数量 | 7 × 10⁹ |
| 梯度大小（fp32） | 约 **28 GB** |
| 一次 AllReduce 每卡搬的数据量 | 约 **49 ~ 56 GB** |

也就是说，**每训练一步，每张卡都要搬几十 GB 的数据**，而且模型越大这个数字线性增长。
这正是为什么「带宽」和「算法」在集群训练里这么要命。

### 2.3 切分模型的情形

切分模型时，一张卡上只有模型的一部分，单卡无法独立完成一次完整的前向 / 反向。
数据必须在承载不同模型分片的卡之间来回流动，通信模式更复杂（见第 3 节）。

### 2.4 关键结论

> **集合通信是一组通信成员都参与的全局通信操作。**

### 2.5 集合通信 vs. 点对点通信

| 维度 | 点对点通信（P2P） | 集合通信（Collective） |
| --- | --- | --- |
| 参与成员 | 2 个通信实体 | 通信域内**全部**成员 |
| 典型操作 | Send / Recv | AllReduce / AllGather / All2All / Broadcast … |
| 语义定义 | 由收发双方约定 | 由通信域内所有成员共同定义，语义统一 |
| 使用方式 | 需要自行编排收发逻辑 | 一次调用即完成全局协同 |
| 适用场景 | 手写通信流程、特定数据搬运 | 并行训练中的梯度同步、参数分发等 |

> **为什么集群训练偏爱集合通信？** 因为训练中的通信需求（梯度同步、参数广播、数据重排）本质
> 就是「全员参与」的。若用点对点通信手工实现，需要编排 $O(n^2)$ 级别的收发逻辑，既易错又低效；
> 集合通信把这种全局协同封装成一次调用，并由库内部针对拓扑做优化。

## 3. 集群训练有哪些常用并行方式

![图 2](./images/png/parallelism_overview.png)

*图 2 · 四类常用并行方式的切分对象与通信需求对照。*

集群训练中的并行方式，本质上都是在回答「把什么切开、切给谁」。
其中前三种属于**模型并行**（Model Parallelism）的范畴。

### 3.1 四类并行方式对照

| 并行方式 | 切分对象 | 做法 | 通信需求 |
| --- | --- | --- | --- |
| **数据并行**<br>Data Parallelism | 数据（Batch） | 不同的 NPU 上执行**同一批数据的不同子集** | 梯度 AllReduce |
| **流水并行**<br>Pipeline Parallelism | 模型（按层） | 不同的 NPU 上执行**模型的不同层** | 层间激活值传递 |
| **张量并行**<br>Tensor Parallelism | 模型（按张量） | **单个运算拆到不同的 NPU 上执行** | 层内张量拼接 / 归约 |
| **专家并行**<br>Expert Parallelism | 模型（按专家） | **把不同的专家放到不同的卡上**（每个专家是一个独立的 FFN） | All2All 分发与回收 |

### 3.2 各方式的侧重点

**① 数据并行**——切数据，不切模型

每张卡持有完整的模型副本，只是喂给它们不同的数据子集。实现最简单，但每张卡都必须装得下
完整模型，因此它的瓶颈仍是**显存容量**。通信上只需要在反向传播后同步梯度。

**② 流水并行**——按层切模型

把模型按层切成若干段，每段放到一张（或一组）卡上，数据像流水线一样依次流过各段。
好处是单卡只需装下几层；代价是引入了**流水线气泡**（bubble）——前面的卡在算时，后面的卡
在等待，需要靠微批（micro-batch）来填充，而微批之间需要传递激活值。

**③ 张量并行**——把一个运算切开，切的是「同一个张量」

张量并行切的是**同一个矩阵运算内部**。以最典型的矩阵乘 $Y = W \cdot X$ 为例：把权重矩阵 $W$
按列（或按行）切成几块，每张卡只持有 $W$ 的一部分，各自算出一部分结果，最后再**拼回**（AllGather）
或**归约**（AllReduce）成完整结果。

关键在于：**每张卡算的是同一层、同一个运算的一部分**——它们算的是「同一件事的不同片段」。
因此切分后必须频繁通信来拼接结果，对带宽和时延要求很高，通常优先放在同机 HCCS 高速互联的卡上。

**④ 专家并行**——把不同的专家分开

专家并行切的是**不同的专家**：**把不同的专家放到不同的卡上，每个专家是一个独立的 FFN**。

关键在于：**每张卡算的是「不同的事」**——卡与卡之间不是同一运算的不同片段，而是各自负责
不同的专家。所以不需要把结果拼回，而是要把 token 发到它该用的专家所在的卡上，算完再送回来；
这个「发过去 + 收回来」用的就是 **All2All**（下一小节举个具体例子）。

> **一张表看清 TP 与 EP 的区别**
>
> | 对比维度 | 张量并行 TP | 专家并行 EP |
> | --- | --- | --- |
> | 切的是什么 | **同一个运算 / 同一个张量**（如 $W$ 的一半） | **不同的专家子网络**（各自完整） |
> | 每张卡算的 | 同一层运算的**一个片段** | **不同的专家**（不同的事） |
> | 卡之间的关系 | 分工协作，共同完成一次运算 | 各管一摊，互不重叠 |
> | 是否需要拼回 | 需要：AllReduce / AllGather 拼回完整结果 | 不需要：结果按 token 各归各位 |
> | 核心通信 | 层内高带宽归约 / 拼接 | **All2All** 分发与回收 |
> | 是否依赖 MoE | 不依赖，任何模型都能用 | 依赖，只用于 MoE 这类稀疏模型 |
>
> 一句话：**TP 是「把一件事拆给多个人做」，EP 是「不同的人负责不同的事」。**

### 3.3 举例：专家并行在 MoE 中怎么工作

上一小节说到：专家并行就是把不同的专家放到不同的卡上，靠 All2All 把 token 发过去、把结果收回来。
这里用一个具体的 MoE 例子把这件事讲透。

![图 3](./images/png/moe_all2all_dispatch_combine.png)

*图 3 · 专家并行里 token 与专家的两次 All2All —— ① Dispatch 把 token 送到「它要用的专家」所在的卡；② Combine 把专家算完的结果送回 token 原本所在的卡。*

**① 从稠密到稀疏**

标准 Transformer 里，每个 token 都要过同一个 FFN，**计算量和参数量绑死**——模型想更大，
计算量就得同比涨。MoE（Mixture-of-Experts）换了个做法：用一组**专家（FFN₁ … FFNₙ）**
加一个**门控（Gating）**，替掉原来那个单独的 FFN。

```
                 ┌─────────────┐
   token ───────►│   Gating    │  为每个 token 选出 Top-K 个专家
                 └──────┬──────┘
                        │  只激活少数专家
             ┌──────────┼──────────┐
             ▼          ▼          ▼
          ┌─────┐    ┌─────┐    ┌─────┐
          │ FFN₁│    │ ... │    │ FFNₙ│
          └──┬──┘    └──┬──┘    └──┬──┘
             └──────────┼──────────┘
                        ▼
                  加权求和输出
```

关键在**稀疏**：专家可以很多（总参数量很大），但每个 token 实际**只激活其中少数几个**，
于是**参数量上去了、计算量却不用同步涨**。

**② 专家放不下，就得分散到不同的卡上**

专家一多，单卡装不下，只能**把不同的专家放到不同的卡上**——这就是专家并行。
可问题也随之而来：**设备 0 上的 token，可能要用设备 1 上的专家来处理。**

**③ 两次 All2All：先把 token 发过去，再把结果送回来**

| 阶段 | 做什么 | 属于哪次 All2All |
| --- | --- | --- |
| **分发 Dispatch** | 各卡按 Gating 结果，把 token 发到「它要用的那个专家」所在的卡 | 第一次 All2All |
| **计算** | 各卡用自己的专家，处理收到的 token | —— |
| **回收 Combine** | 各卡把算完的结果，送回「token 原本所在的卡」 | 第二次 All2All |

> **注意这两次的方向是相反的**：第一次把 token 送出去，第二次把结果送回来。
> 两次都是 All2All（人人发给所有人），但搬的数据不一样。

**④ 为什么这里必须是 All2All**

数据并行的 AllReduce 是「所有卡互相归约**同一份**数据」，最终大家结果相同。
MoE 完全不同：**每张卡要发出去的数据不一样、要收回来的数据也不一样**
（取决于本卡有哪些 token、Gating 把它们分给了谁）。

这种「人人发给所有人、且各发各的」的模式，正是 **All2All** 的语义。它也是 MoE 专家并行
中通信开销最大、最需要优化的环节。

### 3.4 这四者通常组合使用

实际的大模型训练很少只用一种并行方式，而是**混合并行**（Hybrid Parallelism）。一个常见的
组合是：

```
数据并行（跨节点，同步梯度）
  └── 流水并行（节点内按层切分）
        └── 张量并行（层内按张量切分，走 HCCS 高带宽）
              └── 专家并行（MoE 层内的专家分布）
```

选择组合方式时，核心考量是**通信量**与**链路带宽**的匹配：通信越密集的并行方式，越应该放在
带宽越高、时延越低的链路上（例如同机 HCCS 内部）。

> **各类并行方式各用哪些算子、通信量有多大、能否被计算「隐藏」**，见
> **第 2 章 2.2 节「大模型的典型通信模型」**——那里有完整的对照图和三条结论。

## 4. 课堂小练

请根据本节学习内容完成以下题目进行自测。

**1、（判断题）** 只要把模型切分到多张卡上，就不再需要任何卡间通信了。

---

**2、（单选题）** 在数据并行训练中，用于同步各卡梯度、使权重保持一致的核心集合通信操作是：

- **A)** All2All
- **B)** AllReduce
- **C)** Broadcast
- **D)** Send / Recv

---

**3、（单选题）** 稀疏 MoE 模型在专家并行时，用于在设备之间分发与回收 token 的集合通信操作是：

- **A)** AllReduce
- **B)** AllGather
- **C)** All2All
- **D)** ReduceScatter

---

**4、（简答题）** 张量并行（TP）和专家并行（EP）都涉及「切模型」，它们的本质区别是什么？

**执行以下代码获取答案。**

In [5]:
# 读取本节课堂小练答案（用 Python 读文件：跨平台，并保证中文不乱码）
import pathlib
_p = pathlib.Path('./answer/01.02_answer.txt')
if _p.exists():
    print(_p.read_text(encoding='utf-8'))
else:
    print(f'未找到答案文件：{_p}')
    print('（请确认 answer/ 目录与本 notebook 在同一层）')

1.2 集群通信介绍 · 课堂小练答案
1、（判断题）只要把模型切分到多张卡上，就不再需要任何卡间通信了。
答：错误。切分本身就会带来大量数据通信——切数据要对梯度做 AllReduce，切模型要在层间 / 层内
    传递激活值与张量，切 MoE 专家要 All2All 分发与回收。集合通信正是为解决这些需求而存在。
2、（单选题）数据并行训练中，用于同步各卡梯度、使权重保持一致的核心集合通信操作是？
答：B) AllReduce。每张卡用不同数据子集算出各自的梯度，但权重只有一份逻辑副本，必须先把
    多卡梯度归约（求和后取平均）到一致、再广播回所有卡，各卡才能更新出相同的权重。
3、（单选题）稀疏 MoE 模型在专家并行时，用于在设备之间分发与回收 token 的集合通信操作是？
答：C) All2All。共两次：Dispatch 把 token 发到持有目标专家的卡；Combine 把专家算完的结果
    发回 token 原本所在的卡。它与 AllReduce「所有卡互相归约同一份数据」的语义完全不同。
4、（简答题）张量并行（TP）和专家并行（EP）都涉及「切模型」，它们的本质区别是什么？
答：区别在于「切的对象」不同。
    · TP 切的是同一个张量 / 同一个运算（例如把 W 按列切开），每张卡算的是同一层运算的不同
      片段，算完必须用 AllReduce / AllGather 把结果拼回完整——是分工协作完成同一件事。
    · EP 切的是不同的专家（每个专家是一个完整独立的 FFN），每张卡算的是不同的事，不需要
      拼回结果，而是把 token 经 All2All 发到对应专家所在的卡，算完再 All2All 回收。
    一句话：TP 是「把一件事拆给多个人做」，EP 是「不同的人负责不同的事」。



## 5. 本节总结

### 单卡训练的两大瓶颈

- **性能瓶颈**：单卡算力不足以在合理时间内完成训练（25000 卡仍需约 90 天）；
- **内存挑战**：单卡容量装不下模型参数（1.8 T 参数 ≈ 1.8 TB，约需 128 卡）；
- **破局思路**：分布式集群训练 + 并行计算，把数据与模型切分到多卡；
- **新问题**：切分必然带来大量数据通信 —— 这正是集合通信存在的理由；
- **官方数据**：十万卡集群的实际模型算力利用率**仅约 20%**，大量算力在**等通信完成**——
  通信已经成了新的瓶颈。

### 集合通信

- **定义**：一组通信成员都参与的全局通信操作；
- **通信量有多大**：梯度占 4P 字节（P 为参数量）；Ring AllReduce 每卡搬约 **2 倍梯度大小**
  （7B 模型 fp32 → 梯度 28 GB → 每卡一次搬约 **49~56 GB**）；
- **对比点对点**：P2P 只有 2 个成员、语义由双方约定；集合通信是通信域内**全员参与**、
  语义由所有成员共同定义、一次调用完成全局协同。

### 四类并行方式

| 并行方式 | 切什么 | 核心通信 |
| --- | --- | --- |
| 数据并行 DP | 数据 Batch | AllReduce（梯度同步） |
| 流水并行 PP | 模型层 | 层间激活值传递（Send/Recv） |
| 张量并行 TP | **同一个张量运算** | 层内 AllReduce / AllGather |
| 专家并行 EP | **不同的专家（FFN）** | **All2All**（分发 + 回收） |

> 各并行方式的**通信量、节点规模、能否被计算隐藏**，见**第 2 章 2.2 节**。